# LeetCode 788 — Rotated Digits bằng Python

Link đề: https://leetcode.com/problems/rotated-digits/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def rotatedDigits(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một số `x` là **"tốt" (good)** nếu khi **xoay từng chữ số 180 độ** ta được một số **hợp lệ** và **khác `x`**; hãy đếm các số tốt trong đoạn `[1, n]`.
2. Nắm các công cụ Python: `str`, `str.translate`/`dict` ánh xạ chữ số, `set`, `divmod`/`%` và `//` để tách chữ số, `any`/`all`, mảng trạng thái, `functools.lru_cache`, `sum` với generator.
3. Đi từ **mô phỏng bằng chuỗi cho từng số** → **phân loại chữ số bằng tập hợp** → **quy hoạch động tuyến tính `state[x]` từ `state[x // 10]`** → **đếm tổ hợp theo chữ số (`7^m − 3^m`)** → **DP chữ số có nhớ (pos, tight, đã đổi)**.
4. Hiểu mấu chốt: mỗi chữ số thuộc **một trong ba loại** — *đổi sang chữ số khác* (`2,5,6,9`), *giữ nguyên* (`0,1,8`), *không hợp lệ* (`3,4,7`). Một số **tốt ⇔ không chứa chữ số loại "không hợp lệ" và chứa ít nhất một chữ số loại "đổi"**.
5. Tránh các lỗi kinh điển: đếm cả số hợp lệ nhưng **không đổi** (như `1`, `8`, `10`), coi `0,1,8` là không hợp lệ, bỏ qua `3,4,7`, đảo ngược thứ tự chữ số khi xoay, và quên tính chính `n` trong cách đếm theo chữ số.

## 2. Đề bài & ví dụ

Một số nguyên `x` là **tốt (good)** nếu sau khi **xoay 180 độ từng chữ số riêng lẻ**, ta được một số **hợp lệ** và **khác `x`**. Mỗi chữ số **bắt buộc phải được xoay** (không được giữ nguyên tùy ý).

Một số là **hợp lệ** nếu **mọi chữ số sau khi xoay vẫn là chữ số**. Quy tắc:

- `0`, `1`, `8` xoay thành **chính nó**;
- `2` và `5` xoay thành **nhau** (đối xứng gương);
- `6` và `9` xoay thành **nhau**;
- các chữ số còn lại (`3`, `4`, `7`) **không** xoay được thành chữ số nào ⇒ số chứa chúng **không hợp lệ**.

Cho số nguyên `n`, hãy trả về **số lượng số tốt** trong đoạn `[1, n]`.

**Ví dụ 1**

- Input: `n = 10`
- Output: `4`
- Giải thích: trong `[1, 10]` có bốn số tốt: `2, 5, 6, 9`. Các số `1` và `10` hợp lệ nhưng **giữ nguyên** sau khi xoay nên không tốt.

**Ví dụ 2**

- Input: `n = 1`
- Output: `0`

**Ví dụ 3**

- Input: `n = 2`
- Output: `1`

**Ràng buộc:**

- `1 <= n <= 10^4`

Chú ý: ta xoay **từng chữ số một**, **không đảo thứ tự** các chữ số (ví dụ `25` → `52` và `69` → `96`: mỗi chữ số được thay **tại chính vị trí của nó**).

In [1]:
from functools import lru_cache
import random
import time

examples = [(10, 4), (1, 0), (2, 1)]
for n, expected in examples:
    print(f"n = {n} -> kỳ vọng {expected}")

n = 10 -> kỳ vọng 4
n = 1 -> kỳ vọng 0
n = 2 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Số và chuỗi chữ số

Chuyển số sang chuỗi bằng `str(x)`; duyệt từng ký tự là từng chữ số. Chuyển ngược `int(chuỗi)`. Cũng có thể tách chữ số bằng toán học: `x % 10` là chữ số hàng đơn vị, `x // 10` bỏ chữ số cuối.

In [2]:
x = 4096
print(str(x), list(str(x)), int("4096"))
print("x % 10 =", x % 10, "| x // 10 =", x // 10)
digits = []
y = x
while y:
    digits.append(y % 10)
    y //= 10
print("các chữ số từ phải sang trái:", digits)
print("divmod(4096, 10) =", divmod(4096, 10))

4096 ['4', '0', '9', '6'] 4096
x % 10 = 6 | x // 10 = 409
các chữ số từ phải sang trái: [6, 9, 0, 4]
divmod(4096, 10) = (409, 6)


### 3.2. Từ điển ánh xạ và `str.translate`

Một `dict` ánh xạ chữ số → chữ số sau khi xoay. `str.translate` thay thế từng ký tự theo bảng, thao tác một lượt. Ký tự **không có trong bảng** (3, 4, 7) ta để nhận diện riêng là không hợp lệ.

In [3]:
ROT = {"0": "0", "1": "1", "8": "8", "2": "5", "5": "2", "6": "9", "9": "6"}
table = str.maketrans(ROT)
for s in ("2", "10", "69", "25", "1008"):
    print(s, "->", s.translate(table))
print("37 chứa 3 và 7, ví dụ translate (không đổi 3, 7):", "37".translate(table))

2 -> 5
10 -> 10
69 -> 96
25 -> 52
1008 -> 1008
37 chứa 3 và 7, ví dụ translate (không đổi 3, 7): 37


### 3.3. Tập hợp (`set`) và kiểm tra thành viên

`set` cho phép kiểm tra `ch in S` rất nhanh và phép toán tập hợp: `S & T` (giao), `S - T` (hiệu), `S <= T` (tập con). Ta dùng để hỏi "các chữ số của số này có đều thuộc `{0,1,8,2,5,6,9}` không?".

In [4]:
VALID = set("0182569")
SAME = set("018")
for s in ("1008", "25", "37", "10", "69"):
    ds = set(s)
    print(f"{s:>5}: chữ số = {sorted(ds)} | tất cả hợp lệ? {ds <= VALID} | tất cả giữ nguyên? {ds <= SAME}")

 1008: chữ số = ['0', '1', '8'] | tất cả hợp lệ? True | tất cả giữ nguyên? True
   25: chữ số = ['2', '5'] | tất cả hợp lệ? True | tất cả giữ nguyên? False
   37: chữ số = ['3', '7'] | tất cả hợp lệ? False | tất cả giữ nguyên? False
   10: chữ số = ['0', '1'] | tất cả hợp lệ? True | tất cả giữ nguyên? True
   69: chữ số = ['6', '9'] | tất cả hợp lệ? True | tất cả giữ nguyên? False


### 3.4. `any` và `all`

`all(điều_kiện for ... )` đúng nếu **mọi** phần tử thỏa; `any(...)` đúng nếu **có ít nhất một**. Với số: "hợp lệ" là `all` chữ số hợp lệ; "có chữ số đổi" là `any` chữ số thuộc `{2,5,6,9}`.

In [5]:
CHANGE = set("2569")
def classify_str(s):
    if not all(ch in VALID for ch in s):
        return "không hợp lệ"
    return "tốt" if any(ch in CHANGE for ch in s) else "hợp lệ nhưng không đổi"

for s in ("1", "2", "10", "13", "69", "88", "5008"):
    print(f"{s:>5}: {classify_str(s)}")

    1: hợp lệ nhưng không đổi
    2: tốt
   10: hợp lệ nhưng không đổi
   13: không hợp lệ
   69: tốt
   88: hợp lệ nhưng không đổi
 5008: tốt


### 3.5. Mảng trạng thái dựng dần (quy hoạch động tuyến tính)

Ta có thể tính thuộc tính của `x` từ thuộc tính của `x // 10` (bỏ chữ số cuối) và chữ số cuối `x % 10`: `trạng_thái[x] = ghép(trạng_thái[x // 10], chữ_số_cuối)`. Dựng mảng `trạng_thái[0..n]` từ trái sang phải, mỗi phần tử `O(1)`.

### 3.6. Lũy thừa và tổ hợp đếm

Số chuỗi độ dài `m` chọn mỗi vị trí **tự do** trong một tập `A` chữ số là `|A|^m`. Ví dụ có `7` chữ số hợp lệ nên có `7^m` chuỗi độ dài `m` (cho phép số `0` đứng đầu), và `3` chữ số giữ nguyên nên có `3^m` chuỗi chỉ gồm `0,1,8`. Hiệu `7^m − 3^m` đếm chuỗi **hợp lệ và có ít nhất một chữ số đổi**.

In [6]:
for m in range(1, 5):
    print(f"m = {m}: 7^m = {7 ** m:>5}, 3^m = {3 ** m:>3}, hiệu = {7 ** m - 3 ** m:>5}")
print("đếm vét cạn các chuỗi 2 chữ số (kể cả có số 0 đứng đầu):",
      sum(1 for x in range(100) if classify_str(f"{x:02d}") == "tốt"), "= 7^2 - 3^2 =", 7**2 - 3**2)

m = 1: 7^m =     7, 3^m =   3, hiệu =     4
m = 2: 7^m =    49, 3^m =   9, hiệu =    40
m = 3: 7^m =   343, 3^m =  27, hiệu =   316
m = 4: 7^m =  2401, 3^m =  81, hiệu =  2320
đếm vét cạn các chuỗi 2 chữ số (kể cả có số 0 đứng đầu): 40 = 7^2 - 3^2 = 40


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hãy phân loại **từng chữ số** thành ba nhóm:

| nhóm | chữ số | xoay thành | ý nghĩa |
|---|---|---|---|
| **đổi** | `2, 5, 6, 9` | `5, 2, 9, 6` | làm số **khác đi** |
| **giữ nguyên** | `0, 1, 8` | chính nó | hợp lệ nhưng không làm thay đổi |
| **không hợp lệ** | `3, 4, 7` | không có | một chữ số này là số bị loại |

Một số `x` là **tốt** khi và chỉ khi: (1) **không có** chữ số nhóm "không hợp lệ" và (2) **có ít nhất một** chữ số nhóm "đổi" (vì chỉ khi đó số mới khác đi; nếu toàn chữ số "giữ nguyên" thì số y nguyên).

Thử tay với `n = 20`:

| số | chữ số | phân loại | tốt? |
|---|---|---|---|
| 1 | 1 | giữ nguyên | không |
| 2 | 2 | đổi | **có** |
| 3 | 3 | không hợp lệ | không |
| 4, 7 | | không hợp lệ | không |
| 5, 6 | | đổi | **có** |
| 8 | 8 | giữ nguyên | không |
| 9 | 9 | đổi | **có** |
| 10 | 1, 0 | giữ nguyên | không |
| 11 | 1, 1 | giữ nguyên | không |
| 12 | 1, 2 | đổi (có 2), hợp lệ | **có** (`12 → 15`) |
| 13, 14, 17 | | không hợp lệ | không |
| 15, 16, 19 | | có chữ số đổi | **có** |
| 18 | 1, 8 | giữ nguyên | không |
| 20 | 2, 0 | đổi | **có** (`20 → 50`) |

Các số tốt trong `[1, 20]`: `2, 5, 6, 9, 12, 15, 16, 19, 20` ⇒ **9** số.

In [7]:
def is_good_simple(x):
    s = str(x)
    return all(ch in "0182569" for ch in s) and any(ch in "2569" for ch in s)

good = [x for x in range(1, 21) if is_good_simple(x)]
print(good, "->", len(good))

[2, 5, 6, 9, 12, 15, 16, 19, 20] -> 9


## 5. Cách 1: Brute Force (mô phỏng bằng chuỗi cho từng số)

### Ý tưởng

Làm đúng theo đề: với mỗi `x` từ `1` đến `n`:

1. Chuyển `x` thành chuỗi.
2. Nếu có chữ số `3, 4, 7` → **không hợp lệ**, bỏ qua.
3. Xoay từng chữ số theo bảng `0→0, 1→1, 8→8, 2→5, 5→2, 6→9, 9→6` được chuỗi mới.
4. Nếu chuỗi mới **khác** chuỗi cũ → `x` tốt.

In [8]:
ROT = {"0": "0", "1": "1", "8": "8", "2": "5", "5": "2", "6": "9", "9": "6"}


def rotated_digits_brute(n):
    count = 0
    for x in range(1, n + 1):
        s = str(x)
        if any(ch not in ROT for ch in s):        # chữ số 3, 4, 7 -> không hợp lệ
            continue
        rotated = "".join(ROT[ch] for ch in s)
        if rotated != s:
            count += 1
    return count


for n, expected in examples:
    got = rotated_digits_brute(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected
print("n = 20 ->", rotated_digits_brute(20), "| n = 100 ->", rotated_digits_brute(100), "| n = 10^4 ->", rotated_digits_brute(10**4))

10 -> 4 (kỳ vọng 4 )
1 -> 0 (kỳ vọng 0 )
2 -> 1 (kỳ vọng 1 )
n = 20 -> 9 | n = 100 -> 40 | n = 10^4 -> 2320


### Độ phức tạp

- Mỗi số có `O(log n)` chữ số (`≤ 5` với `n ≤ 10^4`), tạo chuỗi và so sánh: Time `O(n log n)`. Với `n = 10^4` chỉ khoảng `5 * 10^4` bước — **rất nhanh**.
- Space `O(log n)` cho chuỗi tạm.

### Nút thắt

Với `n ≤ 10^4` cách này đã đủ để qua bài. Nhưng nó tạo **chuỗi mới cho từng số** và không tận dụng cấu trúc: chữ số của `x` gần như giống `x // 10`. Nếu `n` lên `10^9` hay `10^18` thì không chạy nổi. Hãy xem chi phí tăng thế nào:

In [9]:
for n in (10**3, 10**4, 10**5, 10**6):
    t0 = time.perf_counter()
    r = rotated_digits_brute(n)
    ms = (time.perf_counter() - t0) * 1000
    print(f"n = {n:>8}: {r:>7} số tốt | {ms:8.1f} ms")

n =     1000:     316 số tốt |      0.4 ms
n =    10000:    2320 số tốt |      3.9 ms
n =   100000:   16564 số tốt |     48.5 ms


n =  1000000:  116920 số tốt |    413.0 ms


Thời gian tăng gần **tuyến tính** theo `n`. Để xử lý `n` rất lớn ta cần đếm **không duyệt từng số** (cách 4, 5). Còn cách 2 và 3 giảm hằng số hoặc dùng quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là **tách bài toán theo từng chữ số** thay vì xoay cả số:

> `x` tốt ⇔ **mọi chữ số** thuộc `{0,1,2,5,6,8,9}` **và** **có ít nhất một** chữ số thuộc `{2,5,6,9}`.

Điều này cho nhiều hướng:

- **Cách 2:** kiểm tra từng số bằng **tập hợp** chữ số (không tạo chuỗi xoay).
- **Cách 3:** quy hoạch động theo `x // 10`: `state[x]` chỉ phụ thuộc `state[x // 10]` và chữ số cuối.
- **Cách 4:** **đếm tổ hợp**: `#tốt = #(số chỉ gồm chữ số hợp lệ) − #(số chỉ gồm 0,1,8)` trong `[0, n]`, mỗi vế đếm bằng duyệt chữ số của `n` từ cao xuống thấp (`O(log n)`).
- **Cách 5:** **DP chữ số có nhớ** `(vị trí, còn sát cận không, đã có chữ số đổi chưa)`.

In [10]:
# minh họa: phân loại từng chữ số và kết hợp
def digit_type(d):
    if d in (3, 4, 7):
        return 0      # không hợp lệ
    if d in (0, 1, 8):
        return 1      # giữ nguyên
    return 2          # đổi (2, 5, 6, 9)

print({d: digit_type(d) for d in range(10)})
print("x = 1286:", [digit_type(int(c)) for c in "1286"], "-> có loại 0? có -> không; có loại 2? có -> tốt?")
print("x = 1286 tốt?", is_good_simple(1286), "| x = 1887 tốt?", is_good_simple(1887), "| x = 1801 tốt?", is_good_simple(1801))

{0: 1, 1: 1, 2: 2, 3: 0, 4: 0, 5: 2, 6: 2, 7: 0, 8: 1, 9: 2}
x = 1286: [1, 2, 1, 2] -> có loại 0? có -> không; có loại 2? có -> tốt?
x = 1286 tốt? True | x = 1887 tốt? False | x = 1801 tốt? False


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Phân loại chữ số bằng tập hợp (không tạo chuỗi xoay)

**Giải thích từng đoạn:**

1. Tạo hai tập hằng: `VALID = {0,1,2,5,6,8,9}` (chữ số hợp lệ) và `CHANGE = {2,5,6,9}` (chữ số đổi).
2. Với mỗi `x`, tách chữ số bằng `x % 10`, `x //= 10` (không dùng chuỗi). Gặp chữ số không thuộc `VALID` → loại. Gặp chữ số thuộc `CHANGE` → đánh dấu `changed = True`.
3. Số tốt nếu không bị loại và `changed`.

**Code demo (in cho vài số):**

In [11]:
def good_by_digits_demo(x):
    y, changed, ok = x, False, True
    while y:
        d = y % 10
        if d in (3, 4, 7):
            ok = False
            break
        if d in (2, 5, 6, 9):
            changed = True
        y //= 10
    return ok and changed

for x in (2, 10, 13, 69, 88, 1286, 3000):
    print(f"x = {x:>5}: tốt? {good_by_digits_demo(x)}")
print("n = 20 ->", sum(good_by_digits_demo(x) for x in range(1, 21)))

x =     2: tốt? True
x =    10: tốt? False
x =    13: tốt? False
x =    69: tốt? True
x =    88: tốt? False
x =  1286: tốt? True
x =  3000: tốt? False
n = 20 -> 9


**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    def rotatedDigits(self, n: int) -> int:
        count = 0
        for x in range(1, n + 1):
            y, changed, ok = x, False, True
            while y:
                d = y % 10
                if d in (3, 4, 7):
                    ok = False
                    break
                if d in (2, 5, 6, 9):
                    changed = True
                y //= 10
            if ok and changed:
                count += 1
        return count


for n, expected in examples:
    assert Solution2().rotatedDigits(n) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** Time `O(n log n)` (mỗi số tách `O(log n)` chữ số); Space `O(1)`. Không tạo chuỗi nên hằng số nhỏ hơn cách 1.

### 7.2. Cách 3 — Quy hoạch động tuyến tính: `state[x]` từ `state[x // 10]`

**Ý tưởng khác:** định nghĩa trạng thái của một số `x`:

- `0`: **không hợp lệ** (chứa `3, 4, 7`),
- `1`: hợp lệ và **giữ nguyên** (chỉ gồm `0, 1, 8`),
- `2`: hợp lệ và **tốt** (hợp lệ và có chữ số `2, 5, 6, 9`).

Trạng thái của chữ số `d` cũng thuộc `{0, 1, 2}` (`digit_type`). Ghép hai trạng thái `a` (phần đầu `x // 10`) và `b` (chữ số cuối): nếu `a == 0` hoặc `b == 0` → `0`; ngược lại nếu `a == 2` hoặc `b == 2` → `2`; ngược lại `1`. Quy ước `state[0] = 1` ("số rỗng" giữ nguyên), nên số một chữ số `x` có `state[x] = ghép(1, digit_type(x))` = `digit_type(x)`.

**Giải thích từng đoạn:**

1. `state = [0] * (n + 1)`, `state[0] = 1`.
2. Với `x = 1..n`: `state[x] = ghép(state[x // 10], digit_type(x % 10))`.
3. Đáp án = số `x` với `state[x] == 2`.

**Code demo:**

In [13]:
DIGIT_TYPE = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2]       # chỉ số = chữ số 0..9: 0 hợp lệ-không đổi, 1 giữ, 2 đổi (xem digit_type)
# 0->1, 1->1, 2->2, 3->0, 4->0, 5->2, 6->2, 7->0, 8->1, 9->2

def combine(a, b):
    if a == 0 or b == 0:
        return 0
    if a == 2 or b == 2:
        return 2
    return 1

def dp_states_demo(n):
    state = [0] * (n + 1)
    state[0] = 1
    for x in range(1, n + 1):
        state[x] = combine(state[x // 10], DIGIT_TYPE[x % 10])
    return state

st = dp_states_demo(30)
print("state[0..30] =", st)
print("các số tốt (state == 2):", [x for x in range(1, 31) if st[x] == 2], "-> đếm", sum(1 for x in range(1, 31) if st[x] == 2))

state[0..30] = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2, 1, 1, 2, 0, 0, 2, 2, 0, 1, 2, 2, 2, 2, 0, 0, 2, 2, 0, 2, 2, 0]
các số tốt (state == 2): [2, 5, 6, 9, 12, 15, 16, 19, 20, 21, 22, 25, 26, 28, 29] -> đếm 15


**Code nộp LeetCode (cách 3):**

In [14]:
class Solution3:
    def rotatedDigits(self, n: int) -> int:
        # loại chữ số: 0 = không hợp lệ, 1 = giữ nguyên, 2 = đổi
        kind = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2]
        state = [0] * (n + 1)
        state[0] = 1
        count = 0
        for x in range(1, n + 1):
            a, b = state[x // 10], kind[x % 10]
            if a == 0 or b == 0:
                state[x] = 0
            elif a == 2 or b == 2:
                state[x] = 2
                count += 1
            else:
                state[x] = 1
        return count


for n, expected in examples:
    assert Solution3().rotatedDigits(n) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** Time `O(n)` — mỗi số tốn `O(1)` nhờ dùng kết quả của `x // 10`. Space `O(n)` cho mảng `state` (có thể giảm nếu chỉ cần theo từng tầng, nhưng ở đây không cần).

### 7.3. Cách 4 — Đếm tổ hợp theo chữ số: `7^m − 3^m` kiểu "đếm các số ≤ n có mọi chữ số trong tập A"

**Ý tưởng khác:** `#tốt trong [1, n] = F(A_valid) − F(A_same)` với `A_valid = {0,1,2,5,6,8,9}` (7 chữ số) và `A_same = {0,1,8}` (3 chữ số), trong đó `F(A)` là **số các số nguyên trong `[0, n]` mà mọi chữ số đều thuộc `A`**. Số `0` thuộc cả hai nên triệt tiêu; một số hợp lệ nhưng chỉ gồm `0,1,8` bị trừ đi, đúng loại bỏ số "giữ nguyên".

**Cách tính `F(A)` (duyệt chữ số của `n` từ cao xuống thấp):** gọi `n` có `L` chữ số `d_1 d_2 ... d_L`. Ta đếm số có `L` chữ số (kể cả có số 0 đứng đầu, tức cũng đếm các số ngắn hơn) bằng cách **chọn vị trí đầu tiên mà chữ số nhỏ hơn `d_i`**:

- Với mỗi vị trí `i`: số cách chọn chữ số **nhỏ hơn `d_i`** và thuộc `A` là `less_i = #{a ∈ A : a < d_i}`; các vị trí sau `i` (còn `L - i` vị trí) chọn **tự do** trong `A`: `|A|^(L - i)`. Cộng `less_i * |A|^(L - i)`.
- Nếu `d_i ∉ A` thì **dừng** (không thể giữ sát `n` tiếp).
- Nếu duyệt hết mà mọi `d_i ∈ A` thì **chính `n`** cũng thỏa → cộng 1.

**Code demo (in từng bước):**

In [15]:
def count_upto(n, allowed):
    """Số nguyên trong [0, n] mà mọi chữ số thuộc tập allowed (allowed là set các số nguyên 0..9)."""
    digits = list(map(int, str(n)))
    L, m = len(digits), len(allowed)
    total = 0
    for i, d in enumerate(digits):
        less = sum(1 for a in allowed if a < d)
        add = less * m ** (L - i - 1)
        print(f"   vị trí {i} (chữ số {d}): có {less} chữ số < {d} trong tập -> + {less} * {m}^{L - i - 1} = {add}")
        total += add
        if d not in allowed:
            print(f"   chữ số {d} không thuộc tập -> dừng")
            return total
    print("   mọi chữ số của n đều thuộc tập -> cộng thêm chính n (1)")
    return total + 1

VALID_SET = {0, 1, 2, 5, 6, 8, 9}
SAME_SET = {0, 1, 8}
for n in (10, 25):
    print(f"n = {n}:")
    a = count_upto(n, VALID_SET)
    print("  F(hợp lệ) =", a)
    b = count_upto(n, SAME_SET)
    print("  F(giữ nguyên) =", b)
    print("  số tốt =", a - b)

n = 10:
   vị trí 0 (chữ số 1): có 1 chữ số < 1 trong tập -> + 1 * 7^1 = 7
   vị trí 1 (chữ số 0): có 0 chữ số < 0 trong tập -> + 0 * 7^0 = 0
   mọi chữ số của n đều thuộc tập -> cộng thêm chính n (1)
  F(hợp lệ) = 8
   vị trí 0 (chữ số 1): có 1 chữ số < 1 trong tập -> + 1 * 3^1 = 3
   vị trí 1 (chữ số 0): có 0 chữ số < 0 trong tập -> + 0 * 3^0 = 0
   mọi chữ số của n đều thuộc tập -> cộng thêm chính n (1)
  F(giữ nguyên) = 4
  số tốt = 4
n = 25:
   vị trí 0 (chữ số 2): có 2 chữ số < 2 trong tập -> + 2 * 7^1 = 14
   vị trí 1 (chữ số 5): có 3 chữ số < 5 trong tập -> + 3 * 7^0 = 3
   mọi chữ số của n đều thuộc tập -> cộng thêm chính n (1)
  F(hợp lệ) = 18
   vị trí 0 (chữ số 2): có 2 chữ số < 2 trong tập -> + 2 * 3^1 = 6
   chữ số 2 không thuộc tập -> dừng
  F(giữ nguyên) = 6
  số tốt = 12


**Code nộp LeetCode (cách 4):**

In [16]:
class Solution4:
    def rotatedDigits(self, n: int) -> int:
        def count_upto(limit, allowed):
            digits = list(map(int, str(limit)))
            L, m = len(digits), len(allowed)
            total = 0
            for i, d in enumerate(digits):
                less = sum(1 for a in allowed if a < d)
                total += less * m ** (L - i - 1)
                if d not in allowed:
                    return total
            return total + 1

        return count_upto(n, {0, 1, 2, 5, 6, 8, 9}) - count_upto(n, {0, 1, 8})


for n, expected in examples:
    assert Solution4().rotatedDigits(n) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp:** Time `O(log n)` (hai lần duyệt `L = O(log n)` chữ số, mỗi bước tính `less` trên tập tối đa 7 phần tử). Space `O(log n)` cho danh sách chữ số. Hiện thực này **chạy được cho `n` cực lớn** (ví dụ `10^18`), không bị giới hạn bởi `n ≤ 10^4`.

### 7.4. Cách 5 — DP chữ số có nhớ (vị trí, còn sát cận, đã đổi)

**Ý tưởng khác:** cùng tinh thần đếm theo chữ số nhưng viết thành **khuôn DP chữ số** tổng quát: `dfs(pos, tight, changed)` là số cách điền các chữ số từ vị trí `pos` đến hết sao cho:

- mọi chữ số điền vào thuộc `VALID` (không có `3,4,7`),
- nếu `tight` là `True` thì các chữ số điền **không vượt** chữ số tương ứng của `n` (còn sát cận); khi đã chọn chữ số **nhỏ hơn** thì `tight` thành `False`,
- cuối cùng (`pos == L`) tính 1 nếu `changed` (đã có chữ số `2,5,6,9`), ngược lại 0.

Số `0` được tính vào (toàn chữ số 0 → không đổi → không đếm). Kết quả `dfs(0, True, False)`.

**Giải thích từng đoạn:**

1. `digits = [chữ số của n]`.
2. `dfs(pos, tight, changed)`: nếu `pos == L` → `1 if changed else 0`.
3. `hi = digits[pos] if tight else 9`; duyệt `d` từ `0` đến `hi`: bỏ `d ∈ {3,4,7}`; gọi `dfs(pos + 1, tight and d == hi, changed or d in {2,5,6,9})`.
4. `lru_cache` để nhớ (số trạng thái `L * 2 * 2` rất nhỏ).

**Code demo (đếm số trạng thái và kết quả):**

In [17]:
def digit_dp_demo(n):
    digits = list(map(int, str(n)))
    L = len(digits)

    @lru_cache(maxsize=None)
    def dfs(pos, tight, changed):
        if pos == L:
            return 1 if changed else 0
        hi = digits[pos] if tight else 9
        total = 0
        for d in range(hi + 1):
            if d in (3, 4, 7):
                continue
            total += dfs(pos + 1, tight and d == hi, changed or d in (2, 5, 6, 9))
        return total

    ans = dfs(0, True, False)
    print(f"n = {n}: đáp án {ans}, số trạng thái khác nhau đã tính: {dfs.cache_info().currsize}")
    return ans

for n in (10, 25, 10**4):
    digit_dp_demo(n)

n = 10: đáp án 4, số trạng thái khác nhau đã tính: 6
n = 25: đáp án 12, số trạng thái khác nhau đã tính: 6
n = 10000: đáp án 2320, số trạng thái khác nhau đã tính: 15


**Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    def rotatedDigits(self, n: int) -> int:
        digits = list(map(int, str(n)))
        L = len(digits)

        @lru_cache(maxsize=None)
        def dfs(pos, tight, changed):
            if pos == L:
                return 1 if changed else 0
            hi = digits[pos] if tight else 9
            total = 0
            for d in range(hi + 1):
                if d in (3, 4, 7):
                    continue
                total += dfs(pos + 1, tight and d == hi, changed or d in (2, 5, 6, 9))
            return total

        return dfs(0, True, False)


for n, expected in examples:
    assert Solution5().rotatedDigits(n) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp:** số trạng thái `O(L * 2 * 2)`, mỗi trạng thái thử `≤ 10` chữ số ⇒ Time `O(10 * 4 * L) = O(log n)`; Space `O(log n)`. Khuôn "DP chữ số" này áp dụng cho rất nhiều bài đếm số theo tính chất chữ số (ví dụ LeetCode 233, 902, 1012, 2376).

## 8. TRACE từng bước

### 8.1. Trace cách 1 — từng số từ 1 đến 20

Bảng trạng thái: chuỗi `x`, kết quả xoay (hoặc không hợp lệ), kết luận và đáp án tạm thời.

In [19]:
def trace_brute(n):
    count = 0
    print(f"{'x':>3} | {'xoay':>10} | kết luận | đếm")
    for x in range(1, n + 1):
        s = str(x)
        if any(ch not in ROT for ch in s):
            print(f"{x:>3} | {'(3/4/7)':>10} | không hợp lệ | {count}")
            continue
        r = "".join(ROT[ch] for ch in s)
        if r != s:
            count += 1
            print(f"{x:>3} | {r:>10} | TỐT | {count}")
        else:
            print(f"{x:>3} | {r:>10} | không đổi | {count}")
    return count

print("kết quả:", trace_brute(20))

  x |       xoay | kết luận | đếm
  1 |          1 | không đổi | 0
  2 |          5 | TỐT | 1
  3 |    (3/4/7) | không hợp lệ | 1
  4 |    (3/4/7) | không hợp lệ | 1
  5 |          2 | TỐT | 2
  6 |          9 | TỐT | 3
  7 |    (3/4/7) | không hợp lệ | 3
  8 |          8 | không đổi | 3
  9 |          6 | TỐT | 4
 10 |         10 | không đổi | 4
 11 |         11 | không đổi | 4
 12 |         15 | TỐT | 5
 13 |    (3/4/7) | không hợp lệ | 5
 14 |    (3/4/7) | không hợp lệ | 5
 15 |         12 | TỐT | 6
 16 |         19 | TỐT | 7
 17 |    (3/4/7) | không hợp lệ | 7
 18 |         18 | không đổi | 7
 19 |         16 | TỐT | 8
 20 |         50 | TỐT | 9
kết quả: 9


Quan sát: `12 → 15`, `15 → 12`, `16 → 19`, `19 → 16`, `20 → 50` đều khác số gốc; `10, 11, 18` giữ nguyên; `13, 14, 17` có chữ số `3/4/7`.

### 8.2. Trace cách 3 (DP tuyến tính) — `state[x] = ghép(state[x // 10], kind[x % 10])`

In [20]:
def trace_dp(n):
    kind = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2]
    name = {0: "không hợp lệ", 1: "giữ nguyên", 2: "TỐT"}
    state = [0] * (n + 1)
    state[0] = 1
    count = 0
    for x in range(1, n + 1):
        a, b = state[x // 10], kind[x % 10]
        if a == 0 or b == 0:
            state[x] = 0
        elif a == 2 or b == 2:
            state[x] = 2
            count += 1
        else:
            state[x] = 1
        print(f"x = {x:>2}: state[{x // 10}] = {a} ({name[a]}), chữ số {x % 10} loại {b} -> state[{x}] = {state[x]} ({name[state[x]]}) | đếm = {count}")
    return count

print("kết quả:", trace_dp(21))

x =  1: state[0] = 1 (giữ nguyên), chữ số 1 loại 1 -> state[1] = 1 (giữ nguyên) | đếm = 0
x =  2: state[0] = 1 (giữ nguyên), chữ số 2 loại 2 -> state[2] = 2 (TỐT) | đếm = 1
x =  3: state[0] = 1 (giữ nguyên), chữ số 3 loại 0 -> state[3] = 0 (không hợp lệ) | đếm = 1
x =  4: state[0] = 1 (giữ nguyên), chữ số 4 loại 0 -> state[4] = 0 (không hợp lệ) | đếm = 1
x =  5: state[0] = 1 (giữ nguyên), chữ số 5 loại 2 -> state[5] = 2 (TỐT) | đếm = 2
x =  6: state[0] = 1 (giữ nguyên), chữ số 6 loại 2 -> state[6] = 2 (TỐT) | đếm = 3
x =  7: state[0] = 1 (giữ nguyên), chữ số 7 loại 0 -> state[7] = 0 (không hợp lệ) | đếm = 3
x =  8: state[0] = 1 (giữ nguyên), chữ số 8 loại 1 -> state[8] = 1 (giữ nguyên) | đếm = 3
x =  9: state[0] = 1 (giữ nguyên), chữ số 9 loại 2 -> state[9] = 2 (TỐT) | đếm = 4
x = 10: state[1] = 1 (giữ nguyên), chữ số 0 loại 1 -> state[10] = 1 (giữ nguyên) | đếm = 4
x = 11: state[1] = 1 (giữ nguyên), chữ số 1 loại 1 -> state[11] = 1 (giữ nguyên) | đếm = 4
x = 12: state[1] = 1 (giữ nguy

Ví dụ `x = 12`: `state[1] = 1` (giữ nguyên), chữ số `2` loại `2` (đổi) ⇒ `state[12] = 2` (tốt). `x = 13`: chữ số `3` loại `0` ⇒ `state[13] = 0`. `x = 20`: `state[2] = 2`, chữ số `0` loại `1` ⇒ `state[20] = 2`: ghi nhớ rằng đã có chữ số đổi ở phần đầu.

### 8.3. Trace cách 4 (đếm tổ hợp) với `n = 1286`

Ta chọn `n = 1286` để thấy cả 4 vị trí và việc dừng/không dừng:

In [21]:
def trace_count(n):
    digits = list(map(int, str(n)))
    L = len(digits)
    result = {}
    for name, allowed in (("hợp lệ {0,1,2,5,6,8,9}", VALID_SET), ("giữ nguyên {0,1,8}", SAME_SET)):
        m = len(allowed)
        total = 0
        print(f"--- đếm các số trong [0, {n}] chỉ gồm chữ số thuộc tập {name} (|A| = {m})")
        stopped = False
        for i, d in enumerate(digits):
            less = sum(1 for a in allowed if a < d)
            add = less * m ** (L - i - 1)
            total += add
            print(f"vị trí {i}: chữ số {d}, {less} chữ số nhỏ hơn thuộc tập -> + {less} * {m}^{L - i - 1} = {add}; tổng = {total}")
            if d not in allowed:
                print(f"   {d} không thuộc tập -> dừng")
                stopped = True
                break
        if not stopped:
            total += 1
            print(f"   n = {n} cũng thuộc tập -> + 1; tổng = {total}")
        result[name] = total
    a, b = result.values()
    print(f"số tốt = {a} - {b} = {a - b}")
    return a - b

print("kết quả:", trace_count(1286))
print("đối chiếu vét cạn:", rotated_digits_brute(1286))

--- đếm các số trong [0, 1286] chỉ gồm chữ số thuộc tập hợp lệ {0,1,2,5,6,8,9} (|A| = 7)
vị trí 0: chữ số 1, 1 chữ số nhỏ hơn thuộc tập -> + 1 * 7^3 = 343; tổng = 343
vị trí 1: chữ số 2, 2 chữ số nhỏ hơn thuộc tập -> + 2 * 7^2 = 98; tổng = 441
vị trí 2: chữ số 8, 5 chữ số nhỏ hơn thuộc tập -> + 5 * 7^1 = 35; tổng = 476
vị trí 3: chữ số 6, 4 chữ số nhỏ hơn thuộc tập -> + 4 * 7^0 = 4; tổng = 480
   n = 1286 cũng thuộc tập -> + 1; tổng = 481
--- đếm các số trong [0, 1286] chỉ gồm chữ số thuộc tập giữ nguyên {0,1,8} (|A| = 3)
vị trí 0: chữ số 1, 1 chữ số nhỏ hơn thuộc tập -> + 1 * 3^3 = 27; tổng = 27
vị trí 1: chữ số 2, 2 chữ số nhỏ hơn thuộc tập -> + 2 * 3^2 = 18; tổng = 45
   2 không thuộc tập -> dừng
số tốt = 481 - 45 = 436
kết quả: 436
đối chiếu vét cạn: 436


Cả hai lần đếm đều đi hết 4 chữ số của `1286` (mọi chữ số `1,2,8,6` thuộc tập hợp lệ) và cộng thêm 1 cho `1286` ở lần đầu; lần hai tập `{0,1,8}` không chứa `2` nên **dừng** tại vị trí 1.

### 8.4. Trace cách 5 (DP chữ số) — số trạng thái khác nhau và thứ tự gọi

Ta in cây gọi cho `n = 25` (nhỏ), đánh dấu `tight` và `changed`:

In [22]:
def trace_digit_dp(n):
    digits = list(map(int, str(n)))
    L = len(digits)
    seen = {}

    def dfs(pos, tight, changed, depth=0):
        key = (pos, tight, changed)
        pad = "  " * depth
        if key in seen:
            print(f"{pad}dfs{key} -> {seen[key]} (đã nhớ)")
            return seen[key]
        if pos == L:
            r = 1 if changed else 0
            seen[key] = r
            print(f"{pad}dfs{key} -> {r} (hết chữ số)")
            return r
        hi = digits[pos] if tight else 9
        total = 0
        for d in range(hi + 1):
            if d in (3, 4, 7):
                continue
            total += dfs(pos + 1, tight and d == hi, changed or d in (2, 5, 6, 9), depth + 1)
        seen[key] = total
        print(f"{pad}dfs{key} = {total}")
        return total

    return dfs(0, True, False)

print("kết quả n = 25:", trace_digit_dp(25), "| vét cạn:", rotated_digits_brute(25))

    dfs(2, False, False) -> 0 (hết chữ số)
    dfs(2, False, False) -> 0 (đã nhớ)
    dfs(2, False, True) -> 1 (hết chữ số)
    dfs(2, False, True) -> 1 (đã nhớ)
    dfs(2, False, True) -> 1 (đã nhớ)
    dfs(2, False, False) -> 0 (đã nhớ)
    dfs(2, False, True) -> 1 (đã nhớ)
  dfs(1, False, False) = 4
  dfs(1, False, False) -> 4 (đã nhớ)
    dfs(2, False, True) -> 1 (đã nhớ)
    dfs(2, False, True) -> 1 (đã nhớ)
    dfs(2, False, True) -> 1 (đã nhớ)
    dfs(2, True, True) -> 1 (hết chữ số)
  dfs(1, True, True) = 4
dfs(0, True, False) = 12
kết quả n = 25: 12 | vét cạn: 12


Các dòng "(đã nhớ)" cho thấy trạng thái trùng lặp chỉ tính một lần (đây là phần "quy hoạch động" trong DP chữ số). Với `n = 25` có `L = 2` chữ số nên số trạng thái rất nhỏ.

## 9. Code nộp LeetCode

Với ràng buộc `n ≤ 10^4`, mọi cách đều qua. Ta chọn **cách 3 (DP tuyến tính)**: `O(n)`, đơn giản, không cần toán đặc biệt, và minh họa quy hoạch động; cách 4/5 (`O(log n)`) mạnh hơn nhưng khó hơn cho người mới. Đây là code đúng class/method/signature:

In [23]:
class Solution:
    def rotatedDigits(self, n: int) -> int:
        # loại chữ số: 0 = không hợp lệ (3, 4, 7), 1 = giữ nguyên (0, 1, 8), 2 = đổi (2, 5, 6, 9)
        kind = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2]
        state = [0] * (n + 1)                 # state[x]: 0 không hợp lệ, 1 giữ nguyên, 2 tốt
        state[0] = 1                          # "số rỗng" giữ nguyên
        count = 0
        for x in range(1, n + 1):
            a, b = state[x // 10], kind[x % 10]
            if a == 0 or b == 0:
                state[x] = 0
            elif a == 2 or b == 2:
                state[x] = 2
                count += 1
            else:
                state[x] = 1
        return count


for n, expected in examples:
    got = Solution().rotatedDigits(n)
    print(n, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected
print("n = 10^4 ->", Solution().rotatedDigits(10**4))

10 -> 4 (kỳ vọng 4 )
1 -> 0 (kỳ vọng 0 )
2 -> 1 (kỳ vọng 1 )
n = 10^4 -> 2320


## 10. Vì sao thuật toán đúng

**Đặc trưng số tốt.** Xoay từng chữ số: chữ số `d` thành `rot(d)` ở **cùng vị trí**. Số xoay hợp lệ ⇔ mọi `rot(d)` xác định, tức mọi `d ∈ {0,1,2,5,6,8,9}` (các chữ số `3,4,7` không có ảnh). Khi hợp lệ, số xoay **khác** số gốc ⇔ tồn tại vị trí mà `rot(d) ≠ d` ⇔ tồn tại chữ số `d ∈ {2,5,6,9}` (vì `rot(0)=0`, `rot(1)=1`, `rot(8)=8`). Vậy

> `x` tốt ⇔ (mọi chữ số ∈ `{0,1,2,5,6,8,9}`) **và** (có chữ số ∈ `{2,5,6,9}`).

Hai điều kiện này là cơ sở cho cả 5 cách. Cách 1 kiểm tra **trực tiếp** định nghĩa (xoay rồi so sánh) và tương đương với đặc trưng trên.

**Cách 2.** Kiểm tra đúng hai điều kiện bằng cách tách chữ số.

**Cách 3.** Gọi `s(x) ∈ {0, 1, 2}` như đã định nghĩa. Bằng quy nạp theo số chữ số: `x` có chữ số cuối `b = x % 10` và phần đầu `x // 10` (có thể là số rỗng/0). Mọi chữ số của `x` hợp lệ ⇔ phần đầu hợp lệ và `b` hợp lệ ⇔ `s(x // 10) ≠ 0` và `kind[b] ≠ 0`. Có chữ số đổi ⇔ phần đầu có chữ số đổi hoặc `b` đổi ⇔ `s(x // 10) = 2` hoặc `kind[b] = 2` (khi hợp lệ). Hàm ghép trong code đúng bằng điều này. Khởi tạo `s(0) = 1` (không có chữ số nào ⇒ hợp lệ, không đổi); với `x` một chữ số, `x // 10 = 0` nên `s(x) = kind[x]` ✓. Số `0` đứng đầu không xuất hiện vì ta không viết các số có số 0 đứng đầu.

**Cách 4.** Gọi `F(A)` = số nguyên trong `[0, n]` có mọi chữ số thuộc `A` (viết **không** có số 0 thừa đứng đầu, nhưng ta đếm theo độ dài cố định `L` có đệm 0; mỗi số trong `[0, n]` tương ứng **duy nhất** một chuỗi độ dài `L` có đệm 0 và mọi chữ số 0 đệm thuộc `A` vì `0 ∈ A`). Với tập `A ∋ 0`, số nguyên `x ∈ [0, n]` có mọi chữ số thuộc `A` ⇔ chuỗi đệm 0 của `x` toàn chữ số thuộc `A`. Đếm các chuỗi độ dài `L` nhỏ hơn hoặc bằng `n` (so sánh từ điển) bằng cách phân loại theo **vị trí đầu tiên khác `n`**: tại vị trí `i` chọn chữ số `< d_i` thuộc `A`, các vị trí trước bằng `d_1..d_{i-1}` (cần chúng thuộc `A`, nên ta dừng khi gặp `d_i ∉ A`), các vị trí sau tự do `|A|^(L-i)`; cộng 1 cho chính `n` nếu mọi `d_i ∈ A`. Tính xong `F(valid) − F(same)` bằng số các số có mọi chữ số hợp lệ mà **không** mọi chữ số giữ nguyên, tức đúng số tốt; số `0` ở cả hai vế nên triệt tiêu (và `0` không thuộc `[1, n]`).

**Cách 5.** Là cùng phép đếm theo khuôn đệ quy có nhớ: `tight` đảm bảo các chữ số không vượt `n`; `changed` ghi nhớ đã có chữ số đổi; điều kiện lọc `d ∉ {3,4,7}` đảm bảo hợp lệ; trường hợp `pos == L` chỉ đếm khi `changed`. Nó đếm các chuỗi đệm 0 từ `0` đến `n`; chuỗi toàn 0 có `changed = False` nên không được đếm, khớp với việc chỉ đếm `[1, n]`.

**Trường hợp biên.** `n = 1`: số `1` giữ nguyên → 0. `n = 2`: `2` tốt → 1.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng bằng chuỗi | `O(n log n)` | `O(log n)` |
| 2. Tập hợp chữ số | `O(n log n)` | `O(1)` |
| 3. DP tuyến tính (nộp) | `O(n)` | `O(n)` |
| 4. Đếm tổ hợp theo chữ số | `O(log n)` | `O(log n)` |
| 5. DP chữ số có nhớ | `O(log n)` | `O(log n)` |

Với `n = 10^4` mọi cách chạy trong vài mili-giây. Cách 4 và 5 còn chạy được với `n` khổng lồ:

In [24]:
for n in (10**4, 10**6):
    row, answers = [], set()
    for label, cls in (("cách 1", None), ("cách 2", Solution2), ("cách 3", Solution3), ("cách 4", Solution4), ("cách 5", Solution5)):
        t0 = time.perf_counter()
        r = rotated_digits_brute(n) if cls is None else cls().rotatedDigits(n)
        ms = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{label} {ms:8.2f} ms")
    assert len(answers) == 1
    print(f"n = {n:>8}: đáp án {answers.pop():>7} | " + " | ".join(row))

# n cực lớn: chỉ cách 4 và 5
for n in (10**9, 10**18, 2 * 10**18 + 5):
    a, b = Solution4().rotatedDigits(n), Solution5().rotatedDigits(n)
    assert a == b
    print(f"n = {n}: cách 4 = cách 5 = {a}")

n =    10000: đáp án    2320 | cách 1     4.10 ms | cách 2     2.26 ms | cách 3     0.78 ms | cách 4     0.02 ms | cách 5     0.04 ms


n =  1000000: đáp án  116920 | cách 1   386.15 ms | cách 2   277.54 ms | cách 3    79.43 ms | cách 4     0.07 ms | cách 5     0.08 ms
n = 1000000000: cách 4 = cách 5 = 40333924
n = 1000000000000000000: cách 4 = cách 5 = 1628413210489960
n = 2000000000000000005: cách 4 = cách 5 = 3256826420979924


## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta dựng một **oracle độc lập**: bảng cộng dồn `prefix[x]` = số số tốt trong `[1, x]` tính bằng cách **xoay chuỗi** (cách 1) cho mọi `x ≤ 10^6`, rồi tra cứu. Nhờ vậy ta đối chiếu nhanh trên nhiều `n`.

In [25]:
LIMIT = 10**6
prefix = [0] * (LIMIT + 1)
for x in range(1, LIMIT + 1):
    s = str(x)
    good = all(ch in ROT for ch in s) and "".join(ROT[ch] for ch in s) != s
    prefix[x] = prefix[x - 1] + good
print("oracle dựng xong: prefix[10] =", prefix[10], "| prefix[100] =", prefix[100], "| prefix[10^4] =", prefix[10**4], "| prefix[10^6] =", prefix[10**6])

solvers = [
    ("cách 1", rotated_digits_brute),
    ("cách 2", Solution2().rotatedDigits),
    ("cách 3", Solution3().rotatedDigits),
    ("cách 4", Solution4().rotatedDigits),
    ("cách 5", Solution5().rotatedDigits),
]


def check_all(n, expected=None, skip=()):
    want = prefix[n]
    if expected is not None:
        assert want == expected, (n, want, expected)
    for name, f in solvers:
        if name in skip:
            continue
        assert f(n) == want, (name, n, f(n), want)
    return want


for n, expected in examples:
    check_all(n, expected)
print("ví dụ đề: OK")

edge = [(1, 0), (2, 1), (3, 1), (4, 1), (5, 2), (6, 3), (7, 3), (8, 3), (9, 4), (10, 4),
        (11, 4), (12, 5), (13, 5), (19, 8), (20, 9), (99, None), (100, None), (1000, None), (10**4, None)]
for n, expected in edge:
    check_all(n, expected)
print("edge case: OK")

oracle dựng xong: prefix[10] = 4 | prefix[100] = 40 | prefix[10^4] = 2320 | prefix[10^6] = 116920
ví dụ đề: OK
edge case: OK


Các mốc quan trọng: `n = 1` (không có số tốt), `n = 2` (số tốt đầu tiên), chữ số `3, 4, 7` không làm tăng đáp án (`3 → 1`, `4 → 1`, `7 → 3`), `n = 10` (`10` giữ nguyên), `n = 99/100/1000` (ranh giới độ dài), `n = 10^4` (giới hạn).

### 12.2. Phản ví dụ cho các cách sai

Dựng các cách sai và chứng minh chúng sai bằng ví dụ:

In [26]:
def wrong_count_valid(n):
    """SAI: đếm số hợp lệ (không chứa 3,4,7) mà quên điều kiện 'phải khác đi sau khi xoay'."""
    return sum(1 for x in range(1, n + 1) if all(ch in "0182569" for ch in str(x)))


def wrong_zero_one_eight_invalid(n):
    """SAI: coi 0, 1, 8 là không hợp lệ (chỉ cho phép 2, 5, 6, 9)."""
    return sum(1 for x in range(1, n + 1) if all(ch in "2569" for ch in str(x)))


def wrong_ignore_347(n):
    """SAI: bỏ qua chữ số 3, 4, 7 (coi như giữ nguyên) thay vì loại số."""
    return sum(1 for x in range(1, n + 1) if any(ch in "2569" for ch in str(x)))


def wrong_reverse(n):
    """SAI: xoay cả số như xoay tờ giấy 180 độ (đảo thứ tự chữ số rồi mới đổi)."""
    cnt = 0
    for x in range(1, n + 1):
        s = str(x)
        if all(ch in ROT for ch in s):
            r = "".join(ROT[ch] for ch in reversed(s))
            cnt += r != s
    return cnt


print(f"{'n':>4} | {'đúng':>5} | {'chỉ hợp lệ':>10} | {'0,1,8 loại':>10} | {'bỏ 3,4,7':>8} | {'đảo thứ tự':>10}")
for n in (10, 20, 30, 100):
    print(f"{n:>4} | {prefix[n]:>5} | {wrong_count_valid(n):>10} | {wrong_zero_one_eight_invalid(n):>10} | {wrong_ignore_347(n):>8} | {wrong_reverse(n):>10}")

assert wrong_count_valid(10) != prefix[10]
assert wrong_zero_one_eight_invalid(20) != prefix[20]
assert wrong_ignore_347(30) != prefix[30]
assert wrong_reverse(100) != prefix[100]

   n |  đúng | chỉ hợp lệ | 0,1,8 loại | bỏ 3,4,7 | đảo thứ tự
  10 |     4 |          7 |          4 |        4 |          5
  20 |     9 |         14 |          4 |        9 |         11
  30 |    15 |         20 |          8 |       18 |         16
 100 |    40 |         49 |         20 |       64 |         41


- `chỉ hợp lệ`: đếm cả `1`, `8`, `10`, `11`... (giữ nguyên) ⇒ quá nhiều.
- `0,1,8 loại`: loại mất `12`, `20`, ... nên thiếu.
- `bỏ 3,4,7`: đếm cả `23`, `24`, `27`, ... (có chữ số `2` nhưng cũng có `3/4/7` không hợp lệ) ⇒ thừa.
- `đảo thứ tự`: đề nói xoay **từng chữ số** tại chỗ. Ví dụ `69`: xoay từng chữ số được `96` (khác `69` ⇒ tốt); còn nếu xoay cả tờ giấy (đảo thứ tự rồi đổi `6↔9`) thì `69` lại thành `69` (giống ⇒ không tốt).

### 12.3. Đối chiếu nhiều `n` (toàn bộ nhỏ + ngẫu nhiên có seed)

Kiểm tra **mọi** `n` từ `1` đến `1200` cho cả 5 cách, rồi `n` ngẫu nhiên lớn hơn cho các cách nhanh so với `prefix`.

In [27]:
for n in range(1, 1201):
    check_all(n)
print("mọi n trong [1, 1200]: 5 cách khớp oracle")

rng = random.Random(20788)
for _ in range(60):
    n = rng.randint(1201, 10**4)
    check_all(n)
print("60 n ngẫu nhiên trong [1201, 10^4] (theo ràng buộc đề): 5 cách khớp oracle")

for _ in range(25):
    n = rng.randint(10**4, 10**5)
    check_all(n, skip=("cách 1", "cách 2"))
print("25 n ngẫu nhiên đến 10^5: cách 3, 4, 5 khớp oracle")
for _ in range(300):
    n = rng.randint(10**4, LIMIT)
    check_all(n, skip=("cách 1", "cách 2", "cách 3"))
print("300 n ngẫu nhiên đến 10^6: cách 4, 5 khớp oracle")

# n lớn (ngoài ràng buộc của đề): cách 4 và cách 5 đối chiếu nhau và với phép đếm theo độ dài
for n in [10**k - 1 for k in range(1, 19)] + [10**k for k in range(1, 19)] + [rng.randint(1, 10**18) for _ in range(300)]:
    assert Solution4().rotatedDigits(n) == Solution5().rotatedDigits(n), n
print("n tới 10^18: cách 4 = cách 5")

# công thức đóng cho n = 10^m - 1 (mọi số có tối đa m chữ số): 7^m - 3^m
for m in range(1, 19):
    assert Solution4().rotatedDigits(10**m - 1) == 7**m - 3**m
print("kiểm tra 10^m - 1: số tốt = 7^m - 3^m")

mọi n trong [1, 1200]: 5 cách khớp oracle


60 n ngẫu nhiên trong [1201, 10^4] (theo ràng buộc đề): 5 cách khớp oracle
25 n ngẫu nhiên đến 10^5: cách 3, 4, 5 khớp oracle
300 n ngẫu nhiên đến 10^6: cách 4, 5 khớp oracle
n tới 10^18: cách 4 = cách 5
kiểm tra 10^m - 1: số tốt = 7^m - 3^m


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm cả số hợp lệ nhưng không đổi

**Code sai:** `wrong_count_valid` ở mục 12.2.

In [28]:
print("n = 10, đúng:", Solution().rotatedDigits(10), "| sai:", wrong_count_valid(10), "(đếm cả 1, 8, 10, 0...)")
print("các số bị đếm thừa:", [x for x in range(1, 11) if all(ch in "0182569" for ch in str(x)) and not is_good_simple(x)])

n = 10, đúng: 4 | sai: 7 (đếm cả 1, 8, 10, 0...)
các số bị đếm thừa: [1, 8, 10]


**Vì sao sai:** đề yêu cầu số sau xoay phải **khác** số gốc. Các số chỉ gồm `0, 1, 8` (`1`, `8`, `10`, `11`, `18`, ...) hợp lệ nhưng **không đổi**, nên phải thêm điều kiện "có ít nhất một chữ số `2, 5, 6, 9`".

### Lỗi 2: Coi `0, 1, 8` là không hợp lệ

In [29]:
print("n = 20, đúng:", Solution().rotatedDigits(20), "| sai:", wrong_zero_one_eight_invalid(20))
print("các số tốt bị bỏ sót:", [x for x in range(1, 21) if is_good_simple(x) and not all(ch in "2569" for ch in str(x))])

n = 20, đúng: 9 | sai: 4
các số tốt bị bỏ sót: [12, 15, 16, 19, 20]


**Vì sao sai:** `0, 1, 8` xoay thành chính nó — **hợp lệ**. Chúng chỉ không *góp phần làm khác đi*. Số như `12` (`1` giữ nguyên, `2` đổi thành `5` ⇒ `15`) và `20` (`2` đổi, `0` giữ ⇒ `50`) là **tốt** dù chứa `1`/`0`.

### Lỗi 3: Bỏ qua chữ số `3, 4, 7`

In [30]:
print("n = 30, đúng:", Solution().rotatedDigits(30), "| sai:", wrong_ignore_347(30))
print("các số bị đếm thừa:", [x for x in range(1, 31) if any(ch in "2569" for ch in str(x)) and not is_good_simple(x)])

n = 30, đúng: 15 | sai: 18
các số bị đếm thừa: [23, 24, 27]


**Vì sao sai:** `3, 4, 7` **không xoay được thành chữ số nào** nên làm cả số **không hợp lệ** — dù số đó còn chứa `2, 5, 6, 9`. Ví dụ `23` không tốt (chữ số `3`). Phải **loại** mọi số có chữ số này.

### Lỗi 4: Đảo ngược thứ tự chữ số khi xoay

In [31]:
for x in (69, 25, 12, 16):
    s = str(x)
    per_digit = "".join(ROT[c] for c in s)
    flipped = "".join(ROT[c] for c in reversed(s))
    print(f"x = {x}: xoay từng chữ số -> {per_digit} | đảo thứ tự rồi đổi -> {flipped}")
print("n = 100, đúng:", Solution().rotatedDigits(100), "| sai (đảo):", wrong_reverse(100))

x = 69: xoay từng chữ số -> 96 | đảo thứ tự rồi đổi -> 69
x = 25: xoay từng chữ số -> 52 | đảo thứ tự rồi đổi -> 25
x = 12: xoay từng chữ số -> 15 | đảo thứ tự rồi đổi -> 51
x = 16: xoay từng chữ số -> 19 | đảo thứ tự rồi đổi -> 91
n = 100, đúng: 40 | sai (đảo): 41


**Vì sao sai:** đề nói xoay **từng chữ số riêng lẻ** — mỗi chữ số được thay tại **chính vị trí của nó**, không đảo thứ tự (khác với việc xoay cả tờ giấy 180 độ). Hai cách cho kết quả khác nhau (ví dụ `69`: từng chữ số → `96` ≠ `69` ⇒ tốt; đảo → `69` ⇒ không tốt).

### Lỗi 5: Cách đếm theo chữ số quên cộng chính `n`

In [32]:
def wrong_count_upto(limit, allowed):
    digits = list(map(int, str(limit)))
    L, m = len(digits), len(allowed)
    total = 0
    for i, d in enumerate(digits):
        less = sum(1 for a in allowed if a < d)
        total += less * m ** (L - i - 1)
        if d not in allowed:
            return total
    return total                              # SAI: quên + 1 cho chính limit

def wrong_solution4(n):
    return wrong_count_upto(n, {0, 1, 2, 5, 6, 8, 9}) - wrong_count_upto(n, {0, 1, 8})

for n in (2, 6, 10, 12, 20, 25):
    print(f"n = {n:>2}: đúng {prefix[n]} | quên cộng n: {wrong_solution4(n)}")

n =  2: đúng 1 | quên cộng n: 0
n =  6: đúng 3 | quên cộng n: 2
n = 10: đúng 4 | quên cộng n: 4
n = 12: đúng 5 | quên cộng n: 4
n = 20: đúng 9 | quên cộng n: 8
n = 25: đúng 12 | quên cộng n: 11


**Vì sao sai:** `F(A)` đếm các số trong `[0, n]` **bao gồm chính `n`**. Vòng lặp chỉ cộng các số **nhỏ hơn `n`** (rẽ nhánh sớm); nếu mọi chữ số của `n` thuộc tập thì `n` cũng phải được tính: `+ 1`. Với `n = 2` đúng là `1` (số `2`), còn bản sai bỏ `2`. Hai lần `F` có thể bù trừ nhau trong một số trường hợp (nếu `n` chỉ gồm `0, 1, 8` thì cả hai đều thiếu 1 nên hiệu không đổi), nên lỗi chỉ lộ khi `n` có chữ số đổi như `2, 6, 12`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm các số trong `[1, n]` thỏa tính chất **trên từng chữ số**" → hãy nghĩ **DP chữ số** (digit DP) hoặc **đếm tổ hợp theo từng vị trí** (`|A|^m`); nếu `n` nhỏ (như `10^4`), duyệt từng số cũng đủ.
- **Phân loại chữ số thành vài nhóm** rồi xét điều kiện "không chứa nhóm cấm" và "chứa ít nhất một chữ số nhóm cần": luôn bằng `tất cả − chỉ nhóm không cần`.
- **"Có ít nhất một" = tổng − "không có nào"**: ở bài này `7^m − 3^m`.
- **Khuôn DP chữ số:** `dfs(vị trí, còn_sát_cận, trạng_thái_phụ)`; `tight` chuyển sang `False` khi chọn chữ số nhỏ hơn cận; nhớ bằng `lru_cache`.
- **Xây dần từ `x // 10`:** thuộc tính theo chữ số thường tính được `O(1)` từ `x // 10` — mẫu quy hoạch động tuyến tính (như đếm bit từ `x >> 1`).
- **Đọc kỹ đề:** "xoay từng chữ số" (không đảo), "phải khác số gốc", "mỗi chữ số bắt buộc xoay".
- **Python:** `str(x)`, `divmod`, `set`, `any/all`, `lru_cache`; dùng `10**k` thay vì `pow` khi nhỏ.
- **Test:** dùng oracle độc lập (xoay chuỗi) + bảng cộng dồn để đối chiếu nhanh nhiều `n`; kiểm tra công thức đóng (`7^m − 3^m` cho `10^m − 1`).
- **Họ bài liên quan:** Numbers At Most N Given Digit Set (902), Count Numbers with Unique Digits (357), Strobogrammatic Number (246-248), Confusing Number (1056), Digit Count in Range (233).

## 15. Biến thể

1. **Đếm số tốt trong đoạn `[L, R]`**: `count(R) − count(L − 1)` (xem demo).
2. **Số tốt thứ `k`** (nhỏ thứ `k`): tìm nhị phân trên `n` với hàm đếm (xem bài tập 5).
3. **Confusing Number (LeetCode 1056, Premium):** xoay 180 độ **đảo thứ tự** các chữ số (xoay thật cả số) — khác bài này.
4. **Strobogrammatic Number (LeetCode 246):** số xoay 180 độ **bằng chính nó**.
5. **Numbers At Most N Given Digit Set (LeetCode 902):** đếm số ≤ n chỉ dùng một tập chữ số cho trước — chính là `F(A)` của cách 4.
6. **Thêm chữ số hợp lệ khác:** nếu cho phép thêm bảng xoay khác (ví dụ trong hệ nhị phân/16), chỉ cần đổi bảng loại chữ số.
7. **`n` rất lớn** (`10^18`): chỉ dùng cách 4/5.

Demo biến thể 1 và 4 (đối chiếu với vét cạn):

In [33]:
def count_good_between(L, R):
    return Solution4().rotatedDigits(R) - (Solution4().rotatedDigits(L - 1) if L > 1 else 0)


def is_strobogrammatic(x):
    """Biến thể: số xoay 180 độ (đảo thứ tự + đổi 6<->9) bằng chính nó. Chỉ dùng 0,1,6,8,9."""
    mp = {"0": "0", "1": "1", "8": "8", "6": "9", "9": "6"}
    s = str(x)
    if any(ch not in mp for ch in s):
        return False
    return "".join(mp[c] for c in reversed(s)) == s


print("số tốt trong [10, 30]:", count_good_between(10, 30), "| vét cạn:", sum(is_good_simple(x) for x in range(10, 31)))
print("các số strobogrammatic <= 100:", [x for x in range(0, 101) if is_strobogrammatic(x)])
rng = random.Random(15)
for _ in range(300):
    a, b = sorted((rng.randint(1, 10**4), rng.randint(1, 10**4)))
    assert count_good_between(a, b) == prefix[b] - prefix[a - 1]
print("đoạn [L, R] ngẫu nhiên: OK")
assert [x for x in range(0, 101) if is_strobogrammatic(x)] == [0, 1, 8, 11, 69, 88, 96]

số tốt trong [10, 30]: 11 | vét cạn: 11
các số strobogrammatic <= 100: [0, 1, 8, 11, 69, 88, 96]
đoạn [L, R] ngẫu nhiên: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng chuỗi | Xoay từng số rồi so sánh | `O(n log n)` | `O(log n)` | Dễ hiểu nhất, dùng làm oracle |
| 2. Tập hợp chữ số | Hai điều kiện: hợp lệ + có chữ số đổi | `O(n log n)` | `O(1)` | Hằng số nhỏ hơn, không tạo chuỗi |
| 3. DP tuyến tính | `state[x]` từ `state[x // 10]` | `O(n)` | `O(n)` | **Cách nộp**, `n ≤ 10^4` |
| 4. Đếm tổ hợp theo chữ số | `F(hợp lệ) − F(giữ nguyên)` | `O(log n)` | `O(log n)` | `n` rất lớn; cần công thức |
| 5. DP chữ số có nhớ | `(pos, tight, changed)` | `O(log n)` | `O(log n)` | Khuôn tổng quát cho bài đếm theo chữ số |

Khác nhau thật về ý tưởng: cách 1, 2 **duyệt từng số**; cách 3 **tái sử dụng kết quả của số nhỏ hơn**; cách 4 **đếm trực tiếp bằng tổ hợp** (không liệt kê); cách 5 là phép đếm cùng loại nhưng viết bằng **khuôn đệ quy có nhớ** (tổng quát hơn khi có thêm điều kiện).

In [34]:
# so sánh nhanh trên cùng tập n: đáp án và thời gian
ns = [rng.randint(1, 10**5) for _ in range(40)]
ref = [prefix[n] for n in ns]
for name, f in solvers[1:]:
    t0 = time.perf_counter()
    got = [f(n) for n in ns]
    ms = (time.perf_counter() - t0) * 1000
    assert got == ref
    print(f"{name}: tổng {ms:9.1f} ms cho 40 giá trị n (đến 10^5)")

cách 2: tổng     537.0 ms cho 40 giá trị n (đến 10^5)
cách 3: tổng     153.9 ms cho 40 giá trị n (đến 10^5)
cách 4: tổng       0.2 ms cho 40 giá trị n (đến 10^5)
cách 5: tổng       0.6 ms cho 40 giá trị n (đến 10^5)


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [35]:
print(Solution().rotatedDigits(5))
print(Solution().rotatedDigits(11))
print(Solution().rotatedDigits(100))

2
4
40


**Bài 2 (tính tay).** (a) Liệt kê **tất cả số tốt** trong `[1, 30]` vào `ans2a` (list tăng dần, đang là `None`). (b) Dùng công thức `7^m − 3^m` cho `n = 99` (mọi số tối đa 2 chữ số): số tốt trong `[1, 99]`; gán `ans2b`.

In [36]:
ans2a = None   # TODO: list các số tốt trong [1, 30]
ans2b = None   # TODO: số tốt trong [1, 99]

**Bài 3 (điền TODO).** Hoàn thành `classify(x)` trả về `0` nếu `x` chứa chữ số `3/4/7` (không hợp lệ), `1` nếu hợp lệ nhưng giữ nguyên, `2` nếu tốt (dùng `while x:` và `x % 10`).

In [37]:
def classify(x):
    # TODO
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_rotated_digits(n)` theo cách DP tuyến tính (cách 3) **hoặc** cách đếm theo chữ số (cách 4).

In [38]:
def my_rotated_digits(n):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `kth_good(k)`: số tốt **nhỏ thứ `k`** (`k >= 1`), ví dụ `kth_good(1) = 2`, `kth_good(5) = 12`. (Gợi ý: tìm nhị phân trên `n` sao cho `rotatedDigits(n) >= k`, với cách 4 để đếm nhanh.) (b) Viết `count_good_with_digit_sum(n, s)`: số số tốt `<= n` có **tổng chữ số bằng `s`** (cho `n ≤ 10^4`, vét cạn cũng được).

In [39]:
def kth_good(k):
    # TODO
    pass

def count_good_with_digit_sum(n, s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: tìm các số tốt đến 5: `2, 5` ... Với 11, nhớ `10`, `11` giữ nguyên.
- Bài 2: dùng bảng phân loại ở mục 4; (b) mọi số tối đa 2 chữ số: chuỗi `00..99`.
- Bài 3: duyệt từng chữ số, gặp `3/4/7` trả `0` ngay; có chữ số `2/5/6/9` thì đánh dấu.
- Bài 4: chọn cách 3 nếu chưa quen tổ hợp.
- Bài 5: (a) hàm đếm đơn điệu theo `n`; (b) lọc theo `sum(map(int, str(x)))`.

**Mức 2 (vừa):**
- Bài 2: (a) các số từ 1 đến 20 đã có ở mục 4; với 21..30 hãy loại số có chữ số `3/4/7` và các số chỉ gồm `0,1,8`; (b) `7^2 − 3^2 = 40`.
- Bài 3: `changed = changed or d in (2, 5, 6, 9)`; kết thúc `return 2 if changed else 1`.
- Bài 4 (cách 3): `state[x] = ghép(state[x // 10], kind[x % 10])`.
- Bài 5: (a) `lo, hi = 1, 10`, nhân đôi `hi` cho tới khi đếm `>= k`, rồi tìm nhị phân; (b) kết hợp `is_good_simple`.

**Mức 3 (gần đáp án):**
- Bài 2: (a) kết quả có 15 số: `[2, 5, 6, 9, 12, 15, 16, 19, 20, 21, 22, 25, 26, 28, 29]` — hãy đối chiếu bằng cách chạy `is_good_simple`.
- Bài 3, 4: xem mục 7.2 / 7.3 (code nộp).
- Bài 5: (a) `while lo < hi: mid = (lo + hi) // 2; if Solution4().rotatedDigits(mid) >= k: hi = mid else lo = mid + 1`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [40]:
# Bài 1
assert Solution().rotatedDigits(5) == 2
assert Solution().rotatedDigits(11) == 4
assert Solution().rotatedDigits(100) == 40

# Bài 2
ans2a = [2, 5, 6, 9, 12, 15, 16, 19, 20, 21, 22, 25, 26, 28, 29]
ans2b = 7 ** 2 - 3 ** 2
assert ans2a == [x for x in range(1, 31) if is_good_simple(x)], [x for x in range(1, 31) if is_good_simple(x)]
assert ans2b == Solution().rotatedDigits(99) == 40
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [41]:
# Bài 3
def classify(x):
    changed = False
    while x:
        d = x % 10
        if d in (3, 4, 7):
            return 0
        if d in (2, 5, 6, 9):
            changed = True
        x //= 10
    return 2 if changed else 1

for x in range(1, 3001):
    want = 0 if any(c in "347" for c in str(x)) else (2 if is_good_simple(x) else 1)
    assert classify(x) == want, x
assert sum(1 for x in range(1, 10**4 + 1) if classify(x) == 2) == Solution().rotatedDigits(10**4)
print("Bài 3 OK")

Bài 3 OK


In [42]:
# Bài 4 (DP tuyến tính)
def my_rotated_digits(n):
    kind = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2]
    state = [0] * (n + 1)
    state[0] = 1
    count = 0
    for x in range(1, n + 1):
        a, b = state[x // 10], kind[x % 10]
        if a == 0 or b == 0:
            state[x] = 0
        elif a == 2 or b == 2:
            state[x] = 2
            count += 1
        else:
            state[x] = 1
    return count

for n in range(1, 2001):
    assert my_rotated_digits(n) == prefix[n], n
rng = random.Random(4)
for _ in range(20):
    n = rng.randint(1, 10**4)
    assert my_rotated_digits(n) == prefix[n]
print("Bài 4 OK")

Bài 4 OK


In [43]:
# Bài 5
def kth_good(k):
    lo, hi = 1, 10
    while Solution4().rotatedDigits(hi) < k:
        hi *= 2
    while lo < hi:
        mid = (lo + hi) // 2
        if Solution4().rotatedDigits(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo


def count_good_with_digit_sum(n, s):
    return sum(1 for x in range(1, n + 1) if is_good_simple(x) and sum(map(int, str(x))) == s)


goods = [x for x in range(1, LIMIT + 1) if prefix[x] > prefix[x - 1]]
assert kth_good(1) == 2 and kth_good(5) == 12
for k in list(range(1, 200)) + [1000, 5000, 20000]:
    assert kth_good(k) == goods[k - 1], k
assert count_good_with_digit_sum(20, 5) == sum(1 for x in range(1, 21) if is_good_simple(x) and sum(map(int, str(x))) == 5)
assert count_good_with_digit_sum(10**4, 100) == 0
assert sum(count_good_with_digit_sum(10**4, s) for s in range(0, 46)) == Solution().rotatedDigits(10**4)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP tuyến tính (nộp):**

```python
class Solution:
    def rotatedDigits(self, n: int) -> int:
        kind = [1, 1, 2, 0, 0, 2, 2, 0, 1, 2]   # 0 = không hợp lệ (3,4,7), 1 = giữ nguyên (0,1,8), 2 = đổi (2,5,6,9)
        state = [0] * (n + 1)
        state[0] = 1
        count = 0
        for x in range(1, n + 1):
            a, b = state[x // 10], kind[x % 10]
            if a == 0 or b == 0:
                state[x] = 0
            elif a == 2 or b == 2:
                state[x] = 2
                count += 1
            else:
                state[x] = 1
        return count
```

**Template đếm theo chữ số (n lớn):**

```python
def count_upto(limit, allowed):               # số nguyên trong [0, limit] có mọi chữ số thuộc allowed
    digits = list(map(int, str(limit)))
    L, m, total = len(digits), len(allowed), 0
    for i, d in enumerate(digits):
        total += sum(a < d for a in allowed) * m ** (L - i - 1)
        if d not in allowed:
            return total
    return total + 1
ans = count_upto(n, {0, 1, 2, 5, 6, 8, 9}) - count_upto(n, {0, 1, 8})
```

**Checklist trước khi submit:**

- [ ] Số tốt = **hợp lệ** (không có `3,4,7`) **và** **có ít nhất một** chữ số `2,5,6,9`?
- [ ] `0, 1, 8` được coi là hợp lệ (giữ nguyên), không phải không hợp lệ?
- [ ] Xoay **từng chữ số** tại chỗ, **không đảo** thứ tự?
- [ ] Đoạn đếm là `[1, n]` (không tính 0), có tính chính `n`?
- [ ] DP: `state[0] = 1` (số rỗng giữ nguyên) và ghép trạng thái đúng?
- [ ] Đếm theo chữ số: có `+ 1` cho chính `n` và dừng khi gặp chữ số ngoài tập?
- [ ] Đã so với oracle (xoay chuỗi) trên mọi `n` nhỏ và nhiều `n` ngẫu nhiên?